# MODIS land-surface temperature at the lake

Samples daytime LST from MOD11A1 collection 6.1 (Terra, 1 km) on an 8 x 8 grid of points about 1.1 km apart, centred on the lake, from 24 February 2000 to 1 January 2023. The grid straddles MODIS tiles h23v05 and h24v05, and each point is read from the tile that contains it. Data come from the Microsoft Planetary Computer (no account needed).

**Output** (`Shisper_LST_MODIS`): `lst_lake.csv` (LST in kelvin, 0 = no retrieval, with the decoded QC flags) and `points.csv`. CPU runtime, 20 to 40 minutes; finished years are skipped on a rerun.

## Setup

In [ ]:
%pip install -q pystac-client planetary-computer rasterio

import os, re, time
from concurrent.futures import ThreadPoolExecutor
import numpy as np, pandas as pd
import pystac_client, planetary_computer, rasterio
from rasterio.windows import Window
from rasterio.warp import transform as warp_transform
import matplotlib.pyplot as plt

try:
    from google.colab import drive
    drive.mount('/content/drive')
except ImportError:
    pass

DRIVE = '/content/drive/MyDrive'
OUT = f'{DRIVE}/Shisper_LST_MODIS'
os.makedirs(f'{OUT}/parts', exist_ok=True)

START, END = '2000-02-24', '2023-01-01'
LAKE = (36.36061, 74.56276)                         # lake centre (notebook 01)
N, DLAT, DLON = 8, 0.01, 0.0125                    # 8 x 8 points about 1.1 km apart, one MODIS pixel each
lats = LAKE[0] + (np.arange(N) - (N - 1) / 2) * DLAT
lons = LAKE[1] + (np.arange(N) - (N - 1) / 2) * DLON
PTS = pd.DataFrame([(round(a, 4), round(b, 4)) for a in lats for b in lons], columns=['Latitude', 'Longitude'])
PTS.to_csv(f'{OUT}/points.csv', index=False)

def open_catalog():
    for wait in [30, 60, 120, 240, 300, 0]:        # the Planetary Computer limits request rates
        try:
            return pystac_client.Client.open('https://planetarycomputer.microsoft.com/api/stac/v1')
        except Exception:
            if not wait:
                raise
            time.sleep(wait)
catalog = open_catalog()

## MODIS tiles and pixels of the grid

In [ ]:
def is_terra(it):
    return it.id.upper().startswith('MOD11A1') or str(it.properties.get('platform', '')).lower() == 'terra'

def tile_of(it):
    m = re.search(r'\.(h\d{2}v\d{2})\.', it.id)
    if m:
        return m.group(1)
    p = it.properties
    return f"h{int(p['modis:horizontal-tile']):02d}v{int(p['modis:vertical-tile']):02d}"

def day_of(it):
    m = re.search(r'\.A(\d{4})(\d{3})\.', it.id)
    if m:
        return (pd.Timestamp(f'{m.group(1)}-01-01') + pd.Timedelta(days=int(m.group(2)) - 1)).strftime('%Y-%m-%d')
    return (it.datetime or it.common_metadata.start_datetime).strftime('%Y-%m-%d')

# tiles from the MODIS sinusoidal grid itself: the catalogue footprints are approximate near tile edges
R_, TS_, X0_, Y0_ = 6371007.181, 1111950.5197665233, -20015109.355798, 10007554.677899
lam, phi = np.radians(PTS.Longitude.values), np.radians(PTS.Latitude.values)
h = np.floor((R_ * lam * np.cos(phi) - X0_) / TS_).astype(int)
v = np.floor((Y0_ - R_ * phi) / TS_).astype(int)
NEEDED = sorted({f'h{a:02d}v{b:02d}' for a, b in zip(h, v)})
SBOX = [PTS.Longitude.min() - 2, PTS.Latitude.min() - 1, PTS.Longitude.max() + 2, PTS.Latitude.max() + 1]
print('MODIS tiles:', NEEDED)

def search(a, b):
    for attempt in range(6):
        try:
            return [it for it in catalog.search(collections=['modis-11A1-061'], bbox=SBOX, datetime=f'{a}/{b}',
                                                limit=1000).items() if is_terra(it) and tile_of(it) in NEEDED]
        except Exception:
            if attempt == 5:
                raise
            time.sleep(30 * (attempt + 1))

first = {}
for it in search('2019-06-01', '2019-06-20'):
    first.setdefault(tile_of(it), it)
assert all(t in first for t in NEEDED), f'no scenes for tiles {[t for t in NEEDED if t not in first]}'

def find_key(it, target, fallback):
    for k in it.assets:
        if k.lower() == target:
            return k
    hit = [k for k in it.assets if fallback in k.lower()]
    assert hit, f'no {target} asset in {list(it.assets)}'
    return hit[0]
it0 = next(iter(first.values()))
K_LST, K_QC = find_key(it0, 'lst_day_1km', 'lst_day'), find_key(it0, 'qc_day', 'qc_day')

# each point belongs to the tile that contains it; its row and column stay fixed for all days
TILES, owner = {}, np.array([''] * len(PTS), dtype=object)
for t, it in sorted(first.items()):
    with rasterio.open(planetary_computer.sign(it.assets[K_LST].href)) as src:
        xs, ys = warp_transform('EPSG:4326', src.crs, PTS.Longitude.tolist(), PTS.Latitude.tolist())
        rc = np.array([src.index(x, y) for x, y in zip(xs, ys)], dtype=int)
        inside = (rc[:, 0] >= 0) & (rc[:, 0] < src.height) & (rc[:, 1] >= 0) & (rc[:, 1] < src.width) & (owner == '')
    if inside.any():
        idx = np.nonzero(inside)[0]
        owner[idx] = t
        rows, cols = rc[idx, 0], rc[idx, 1]
        r0, c0 = int(rows.min()), int(cols.min())
        TILES[t] = dict(idx=idx, rows=rows, cols=cols, r0=r0, c0=c0,
                        win=Window(c0, r0, int(cols.max()) - c0 + 1, int(rows.max()) - r0 + 1))
assert (owner != '').all(), 'some grid points are not inside any MODIS tile'
NPIX = sum(len(set(zip(g['rows'].tolist(), g['cols'].tolist()))) for g in TILES.values())
print(' | '.join(f'{t}: {len(g["idx"])} points' for t, g in TILES.items()), '| distinct MODIS pixels:', NPIX)

## LST and QC for every day

In [ ]:
MODLAND = {0: 'LST produced, good quality, not necessary to examine more detailed QA',
           1: 'LST produced, other quality, recommend examination of more detailed QA',
           2: 'LST not produced due to cloud effects',
           3: 'LST not produced primarily due to reasons other than cloud'}
DQ = {0: 'Good data quality', 1: 'Other quality data', 2: 'TBD', 3: 'TBD'}
EMIS = {0: 'Average emissivity error <= 0.01', 1: 'Average emissivity error <= 0.02',
        2: 'Average emissivity error <= 0.04', 3: 'Average emissivity error > 0.04'}
LSTE = {0: 'Average LST error <= 1K', 1: 'Average LST error <= 2K', 2: 'Average LST error <= 3K', 3: 'Average LST error > 3K'}
COLS = ['Latitude', 'Longitude', 'Date', 'MODIS_Tile', 'MOD11A1_061_Line_Y_1km', 'MOD11A1_061_Sample_X_1km',
        'MOD11A1_061_LST_Day_1km', 'MOD11A1_061_QC_Day', 'MOD11A1_061_QC_Day_bitmask', 'MOD11A1_061_QC_Day_MODLAND',
        'MOD11A1_061_QC_Day_MODLAND_Description', 'MOD11A1_061_QC_Day_Data_Quality_flag',
        'MOD11A1_061_QC_Day_Data_Quality_flag_Description', 'MOD11A1_061_QC_Day_Emis_Error_flag',
        'MOD11A1_061_QC_Day_Emis_Error_flag_Description', 'MOD11A1_061_QC_Day_LST_Error_Flag',
        'MOD11A1_061_QC_Day_LST_Error_Flag_Description']
ENV = dict(GDAL_DISABLE_READDIR_ON_OPEN='EMPTY_DIR', GDAL_HTTP_MAX_RETRY='5', GDAL_HTTP_RETRY_DELAY='2')

def read_one(it):
    g, out = TILES[tile_of(it)], {}
    with rasterio.Env(**ENV):
        for key in (K_LST, K_QC):
            for attempt in range(5):
                try:
                    with rasterio.open(planetary_computer.sign(it.assets[key].href)) as src:
                        a = src.read(1, window=g['win'])
                    break
                except Exception:
                    try:
                        planetary_computer.sas.TOKEN_CACHE.clear()    # expired access token
                    except Exception:
                        pass
                    if attempt == 4:
                        return it, None
                    time.sleep(10 * (attempt + 1))
            out[key] = a[g['rows'] - g['r0'], g['cols'] - g['c0']]
    return it, out

years = list(range(int(START[:4]), int(END[:4]) + 1))
t0 = time.time()
for yr in years:
    part = f'{OUT}/parts/lst_{yr}.csv'
    if os.path.exists(part):
        continue
    its = [it for it in search(max(START, f'{yr}-01-01'), min(END, f'{yr}-12-31')) if tile_of(it) in TILES]
    its = list({(day_of(it), tile_of(it)): it for it in its}.values())        # one scene per day and tile
    recs, failed = [], 0
    with ThreadPoolExecutor(6) as pool:
        for it, res in pool.map(read_one, its):
            if res is None:
                failed += 1
                continue
            d, t = day_of(it), tile_of(it)
            g = TILES[t]
            dn = res[K_LST].astype('float64'); qc = res[K_QC].astype(int)
            lst = np.where(dn >= 7500, dn * 0.02, np.where((dn >= 150) & (dn <= 400), dn, 0.0))   # scale 0.02 K; 0 = none
            for k, p in enumerate(g['idx']):
                q = int(qc[k]); ml, dq, em, le = q & 3, (q >> 2) & 3, (q >> 4) & 3, (q >> 6) & 3
                recs.append((PTS.Latitude[p], PTS.Longitude[p], d, t, float(g['rows'][k]), float(g['cols'][k]),
                             round(float(lst[k]), 2), float(q), f'0b{q:08b}', f'0b{ml:02b}', MODLAND[ml],
                             f'0b{dq:02b}', DQ[dq], f'0b{em:02b}', EMIS[em], f'0b{le:02b}', LSTE[le]))
    if failed > max(3, 0.02 * len(its)):
        print(f'  {yr}: {failed} of {len(its)} reads failed, year not saved; run this cell again', flush=True)
        continue
    pd.DataFrame(recs, columns=COLS).to_csv(part, index=False)
    print(f'  {yr}: {len({day_of(it) for it in its})} days | {(time.time() - t0) / 60:.1f} min', flush=True)

missing = [yr for yr in years if not os.path.exists(f'{OUT}/parts/lst_{yr}.csv')]
assert not missing, f'years {missing} are not finished; run this cell again'
ALL = pd.concat([pd.read_csv(f'{OUT}/parts/lst_{yr}.csv') for yr in years], ignore_index=True)
ALL = ALL.drop_duplicates(['Latitude', 'Longitude', 'Date']).sort_values(['Latitude', 'Longitude', 'Date']).reset_index(drop=True)
ALL.to_csv(f'{OUT}/lst_lake.csv', index=False)
print('saved lst_lake.csv,', len(ALL), 'rows')

## Check

In [ ]:
ALL['Date'] = pd.to_datetime(ALL['Date'])
ok = ALL['MOD11A1_061_LST_Day_1km'] > 0
daily = ALL[ok].groupby('Date')['MOD11A1_061_LST_Day_1km'].mean()
s22 = daily[(daily.index >= '2022-01-01') & (daily.index < '2023-01-01')]
fig, ax = plt.subplots(1, 2, figsize=(14, 4))
ax[0].plot(s22.index, s22.values - 273.15, '.', ms=4); ax[0].axhline(0, c='r', lw=.8)
ax[0].set_title('2022, daily mean of cloud-free pixels (°C)')
for t in TILES:
    sel = owner == t
    ax[1].scatter(PTS.Longitude[sel], PTS.Latitude[sel], s=14, label=f'MODIS tile {t}')
ax[1].plot(LAKE[1], LAKE[0], '*', ms=15, c='gold', mec='k', label='lake')
ax[1].legend(fontsize=8, loc='upper left', bbox_to_anchor=(1.02, 1))
ax[1].set_title('8 x 8 sampling grid'); ax[1].set_aspect(1 / np.cos(np.radians(LAKE[0])))
plt.tight_layout(); plt.savefig(f'{OUT}/lst_check.png', dpi=120); plt.show()
print('points:', len(PTS), '| distinct pixels:', NPIX, '| dates:', ALL.Date.min().date(), 'to', ALL.Date.max().date(),
      f'| cloud-free pixel-days: {ok.mean():.1%}')